# Imports and Definitions

In [ ]:
# from google.colab import drive
import os
from astropy.io import fits
from astropy.wcs import WCS
import numpy as np
import glob
import matplotlib.pyplot as plt
from astropy.time import Time
import astropy.units as u
from astropy.coordinates import SpectralCoord, EarthLocation, SkyCoord
from datetime import datetime, timedelta
from scipy.interpolate import interp1d

# drive.mount('/content/drive', force_remount=True)

# ----------------------------
# BeSSSpectra: WCS-free
# ----------------------------
def BeSSSpectra(file):
    """
    Load a 1-D BeSS spectrum from FITS.

    Returns:
        wavelengths: np.ndarray (linear solution from header)
        spectrum: np.ndarray
        header: FITS header
    """
    try:
        f = fits.open(file)
        header = f[0].header
        spectrum = f[0].data

        # Linear wavelength solution from header
        crval1 = header.get('CRVAL1', 6563.0)  # default near Hα
        cdelt1 = header.get('CDELT1', 1.0)
        crpix1 = header.get('CRPIX1', 1.0)

        wavelengths = crval1 + (np.arange(len(spectrum)) + 1 - crpix1) * cdelt1

        return wavelengths, spectrum, header

    except Exception as e:
        print(f"⚠️ Failed to read {file}: {e}")
        return None, None, None

# ----------------------------
# sort_files: wavelength filter + DATE-OBS from header
# ----------------------------
def sort_files(file_paths, spec_feature=6562.8):
    """
    Collect FITS files covering Hα, store observation dates from header.

    Returns:
        valid_files: list of paths
        valid_times: list of astropy Time objects (from DATE-OBS)
    """
    files = glob.glob(file_paths)
    valid_files = []
    valid_times = []

    print(f"Found {len(files)} files in directory.")

    for file in files:
        try:
            wavelengths, spectrum, header = BeSSSpectra(file)
            min = np.min(wavelengths)
            max = np.max(wavelengths)
            if wavelengths is None:
                continue

            # Wavelength coverage filter
            if min > spec_feature or max < spec_feature or np.abs(min - spec_feature) > 1500 or np.abs(max - spec_feature) > 1500:
                continue

            # Get observation date from header
            date_obs = header.get('DATE-OBS', None)
            if date_obs is None:
                print(f"⚠️ {file} has no DATE-OBS")
                continue

            try:
                date_obj = Time(date_obs, format='isot', scale='utc')
            except Exception:
                # Fallback for non-standard DATE-OBS
                date_obj = Time(datetime.strptime(date_obs[:10], "%Y-%m-%d"))

            valid_files.append(file)
            valid_times.append(date_obj)

        except Exception as e:
            print(f"⚠️ Skipping {file} due to error: {e}")
            continue

    # Sort by date
    if valid_files:
        sorted_pairs = sorted(zip(valid_times, valid_files))
        valid_times, valid_files = zip(*sorted_pairs)
    else:
        valid_times, valid_files = [], []

    print(f"{len(valid_files)} files with spectra on {spec_feature} Å and have DATE-OBS.")

    return list(valid_files), list(valid_times)

# ----------------------------
# spec_grid: extract spectra around Hα
# ----------------------------
def spec_grid(target, radius, bess_files_sorted, spec_feature=6562.8):
    """
    Extract spectral segments around Hα from 1-D spectra.

    Args:
        target: placeholder, not used here
        radius: number of pixels on either side of target wavelength
        bess_files_sorted: list of FITS file paths
        spec_feature: target rest wavelength

    Returns:
        result: concatenated np.ndarray of segments
    """
    result = np.array([])

    for fname in bess_files_sorted:
        wavelengths, spectrum, header = BeSSSpectra(fname)
        if wavelengths is None:
            continue

        # Find closest pixel to Hα
        index = np.argmin(np.abs(wavelengths - spec_feature))
        if index - radius < 0 or index + radius > len(spectrum):
            continue

        segment = spectrum[index - radius:index + radius]
        result = np.append(result, segment)

    return result


In [ ]:
# Define plotting functions

def plot_bess_spectra(bess_files_sorted, target_name, offset=0.2, wavelength_range=(6510, 6624), y_shift = 2, save_fig = True):
    """
    Plot stacked BeSS spectra, vertically offset by observation date.

    Parameters
    ----------
    bess_files_sorted : list of str
        Paths to FITS spectra (ideally already filtered by CRVAL1 etc.).
    target_name : str
        Target name, used for plot title and output filename.
    offset : float
        Vertical spacing between spectra.
    wavelength_range : tuple
        (min, max) wavelength range in Å for x-axis.
    """
    all_spectra = []
    all_dates = []

    # --- Collect spectra and observation dates ---
    for fname in bess_files_sorted:
        try:
            wavelengths, spectrum, header = BeSSSpectra(fname)
            date_obs = header.get('DATE-OBS', None)

            # Parse DATE-OBS safely
            if date_obs:
                try:
                    date_obj = datetime.strptime(date_obs[:10], "%Y-%m-%d")
                except ValueError:
                    date_obj = None
            else:
                date_obj = None

            all_spectra.append({
                "file": fname,
                "wave": wavelengths,
                "flux": spectrum / np.nanmax(spectrum),
                "date": date_obj,
                "date_str": date_obs if date_obs else "Unknown"
            })
        except Exception as e:
            print(f"⚠️ Skipping {fname}: {e}")

    # --- Sort by observation date (oldest first) ---
    all_spectra = sorted(all_spectra, key=lambda x: x["date"] or datetime.min)
    dates = [s["date_str"] for s in all_spectra]

    # --- Plot spectra ---
    plt.figure(figsize=(10, 0.5 * len(all_spectra) + 5))
    for j, spec in enumerate(all_spectra):
        plt.plot(spec["wave"], spec["flux"] + (j-y_shift) * offset, color='black', lw=0.8)

    # --- Labeling and aesthetics ---
    plt.xlim(*wavelength_range[0])
    plt.xlabel("Wavelength (Å)")
    plt.ylabel("Observation Date + Normalized Flux Offset")
    plt.title(f"Stacked BeSS Spectra for {target_name} ({len(dates)} Spectra)")

    yticks = np.arange(len(all_spectra)) * offset
    plt.yticks(yticks, dates)

    plt.tight_layout()

    if save_fig:

        # --- Save figure ---
        out_folder = os.path.expanduser(
            f'~/gdrive/Shared drives/MACRO-Be/resources/BeSS Variability Plots/{target_name}/')
        
        if os.path.isdir(out_folder):
            print('')
        else:
            os.mkdir(out_folder)

        out_path = out_folder + f'{target_name}_BeSS_{wavelength_range[1]}.png'

        plt.savefig(out_path, dpi=300)
        print(f"✅ Saved plot to: {out_path}")

        plt.show()
    else:
        plt.show()

def plot_bess_dynamic_spectrum(
    bess_files_sorted,
    target_name,
    wavelength_range=(6510, 6624),
    date_range=None,
    cmap='turbo',
    save_fig = True
):
    """
    Plot dynamic spectrum (time vs wavelength) for BeSS data.
    """

    all_dates = []
    all_fluxes = []

    # --- define common wavelength grid ---
    wmin, wmax = wavelength_range[0]
    common_wave = np.linspace(wmin, wmax, 1000)

    # --- parse date range if provided ---
    if date_range is not None:
        try:
            start_date = datetime.strptime(date_range[0], "%Y-%m-%d")
            end_date = datetime.strptime(date_range[1], "%Y-%m-%d")
        except Exception:
            raise ValueError("❌ date_range must be a tuple of ('YYYY-MM-DD', 'YYYY-MM-DD').")
    else:
        start_date = end_date = None

    # --- read each spectrum ---
    for fname in bess_files_sorted:
        try:
            wave, flux, header = BeSSSpectra(fname)
            date_obs = header.get('DATE-OBS', None)
            if not date_obs:
                continue

            try:
                date_obj = datetime.strptime(date_obs[:10], "%Y-%m-%d")
            except ValueError:
                continue

            # --- filter by date range ---
            if date_range is not None:
                if date_obj < start_date or date_obj > end_date:
                    continue

            # --- restrict to wavelength range ---
            mask = (wave >= wmin) & (wave <= wmax)
            if np.sum(mask) < 10:
                continue

            # --- normalize + interpolate onto common grid ---
            flux = flux / np.nanmax(flux)
            interp_flux = interp1d(
                wave[mask], flux[mask], kind='linear',
                bounds_error=False, fill_value=np.nan
            )(common_wave)

            all_dates.append(date_obj)
            all_fluxes.append(interp_flux)

        except Exception as e:
            print(f"⚠️ Skipping {fname}: {e}")
            continue

    if not all_fluxes:
        print("❌ No valid spectra found in the given date/wavelength range.")
        return

    # --- sort by date ---
    sorted_idx = np.argsort(all_dates)
    dates = [all_dates[i] for i in sorted_idx]
    flux_arrays = [all_fluxes[i] for i in sorted_idx]

    # --- make flux matrix ---
    flux_matrix = np.vstack(flux_arrays)

    # --- Create Y values corresponding to index positions ---
    y_positions = np.arange(len(dates))

    # --- Plot ---
    plt.figure(figsize=(10, 5 + 0.25*len(dates)))
    plt.imshow(
        flux_matrix,
        aspect='auto',
        cmap=cmap,
        extent=[common_wave[0], common_wave[-1], y_positions[0], y_positions[-1]],
        origin='lower'
    )

    # --- Properly aligned y-axis ticks ---
    plt.yticks(y_positions, [d.strftime("%Y-%m-%d") for d in dates])

    plt.xlabel("Wavelength (Å)")
    plt.ylabel("Observation Date")
    plt.title(f"Dynamic Spectrum for {target_name} ({len(dates)} Spectra)")
    cbar = plt.colorbar(label="Normalized Flux")

    plt.tight_layout()

    if save_fig:

        # --- Save plot ---
        out_folder = os.path.expanduser(
            f'~/gdrive/Shared drives/MACRO-Be/resources/BeSS Variability Plots/{target_name}/')
        
        if os.path.isdir(out_folder):
            print('')
        else:
            os.mkdir(out_folder)
        
        out_path = out_folder + f'{target_name}_BeSS_{wavelength_range[1]}_dynamic.png'
        
        plt.savefig(out_path, dpi=300)
        print(f"✅ Saved plot to: {out_path}")

        plt.show()
    else:

        plt.show()

# Get Target Info and Data

In [ ]:
# Specify target of interest

# Primary Targets
# target_name = '69_Ori'
# target_name = 'Pi_Aqr'
# target_name = 'AX_Mon'
# target_name = 'QQ_Gem'
# target_name = 'HD_44637'
# target_name = 'V1369_Ori'
# target_name = 'Bet_CMi'
# target_name = 'CT_Cam'
# target_name = 'PZ_Gem'
# target_name = 'BN_Gem'
# target_name = 'V742_Cas'

# target_name = 'Zet_Tau'
target_name = 'Lam_Eri'
# target_name = 'alpha_eri'
# target_name = 'omi_and'

# Survey Targets

# target_name = 'HD_225095'
# target_name = 'V442_And'


# target_name = '10_Cas'
# target_name = 'HD_628'
# target_name = 'phi_And'
# target_name = 'HD_9709'
# target_name = 'HD 13669'
# target_name = 'V787_Cas'
# target_name = 'tet_Ari'
# target_name = 'HD 20017'
# target_name = 'V415_Aur'

In [ ]:
# os.listdir(os.path.expanduser('~/gdrive/Shared drives/MACRO-Be/resources/RLMT-BeSS-Comparison/BeSS_Data/'))
os.listdir(os.path.expanduser('~/gdrive/Shared drives/MACRO-Be/resources/'))

In [ ]:
# define location of historic fits files
file_paths = os.path.expanduser('~/gdrive/Shared drives/MACRO-Be/resources/RLMT-BeSS-Comparison/BeSS_Data/') + target_name + '/*.fits'

# set pixel radius
radius = 250

Ha = 6562.8
HeI = 6678.2

bess_files_sorted, bess_mjd_sorted = sort_files(file_paths, spec_feature = Ha) # For H-alpha
# bess_files_sorted, bess_mjd_sorted = sort_files(file_paths, spec_feature = HeI) # For Helium I
# result = spec_grid(target, radius, bess_files_sorted)

with fits.open(bess_files_sorted[0], ignore_missing_end = True) as hdul:
    hdu = hdul[0].header
    date_start = hdu.get('DATE-OBS')

with fits.open(bess_files_sorted[len(bess_files_sorted) - 1], ignore_missing_end = True) as hdul:
    hdu = hdul[0].header
    date_end = hdu.get('DATE-OBS')

print('')
print(f'{len(bess_files_sorted)} spectra from {date_start} to {date_end}')

In [ ]:
from astroquery.simbad import Simbad

# Reset to default fields
Simbad.reset_votable_fields()

# List available votable fields
print("Available votable fields:")
print(Simbad.list_votable_fields())

Simbad.add_votable_fields('rvz_radvel', 'ra', 'dec', 'plx_value')

result_table = Simbad.query_object(target_name)

if result_table is None:
    print(f"No results found for {target_name}.")
else:
    print("Returned columns:", result_table.colnames)

    if 'rvz_radvel' in result_table.colnames:
        rv = result_table['rvz_radvel'][0]
        if rv is None:
            print('')
            print(f"No radial (recessional) velocity available for {target_name}.")
            print('')
        else:
            print('')
            print(f"The radial (recessional) velocity of {target_name} is {rv} km/s.")
            print('')
    else:
        print("Radial velocity column not found in the result. Available columns are:")
        print(result_table.colnames)
    if 'ra' in result_table.colnames:
        ra = result_table['ra'][0]
        if ra is None:
            print('')
            print(f"No right ascension available for {target_name}.")
            print('')
        else:
            print('')
            print(f"The right ascension of {target_name} is {ra}.")
            print('')
    else:
        print("Right ascension column not found in the result. Available columns are:")
        print(result_table.colnames)
    if 'dec' in result_table.colnames:
        dec = result_table['dec'][0]
        if dec is None:
            print('')
            print(f"No declination available for {target_name}.")
            print('')
        else:
            print('')
            print(f"The declination of {target_name} is {dec}.")
            print('')
    else:
        print("Declination column not found in the result. Available columns are:")
        print(result_table.colnames)
    if 'plx_value' in result_table.colnames:
        plx = result_table['plx_value'][0]
        if plx is None:
            print('')
            print(f"No parallax available for {target_name}.")
            print('')
        else:
            print('')
            print(f"The parallax of {target_name} is {plx} mas.")
            print('')
    else:
        print("Parallax column not found in the result. Available columns are:")
        print(result_table.colnames)

# Get Target Info

target = SkyCoord(ra = ra*u.deg, dec =  dec*u.deg, frame='icrs',
                  radial_velocity=rv*u.km / u.s,
                  distance=1/((plx)*10**(-3)) * u.pc)


In [ ]:
# Old FITS files have old outdated keywords in the header, this block is to supress those warnings crowding the output

# import warnings
# import logging

# # --- Suppress Astropy Warnings ---
# from astropy.wcs import FITSFixedWarning
# from astropy.utils.exceptions import AstropyWarning
# warnings.filterwarnings('ignore', category=FITSFixedWarning)
# warnings.filterwarnings('ignore', category=AstropyWarning)
# warnings.filterwarnings('ignore', message='RADECSYS')
# warnings.filterwarnings('ignore', message='Coordinates system')

# # --- Suppress NoVelocityWarning (from SpectralCoord) ---
# from astropy.coordinates.spectral_coordinate import NoVelocityWarning
# warnings.filterwarnings('ignore', category=NoVelocityWarning)

# # --- Suppress Astroquery's logging warnings ---
# logging.getLogger('astroquery').setLevel(logging.ERROR)
# logging.getLogger('astropy').setLevel(logging.ERROR)

In [ ]:
# Debugging block
# 
#  for fname in bess_files_sorted:
#         wavelengths, spectrum, header = BeSSSpectra(fname)
#         print(f'Wavlength Range for {fname}: {np.min(wavelengths)} to {np.max(wavelengths)} (Å)')

# Plot Data

In [ ]:
print(f'{len(bess_files_sorted)} spectra from {date_start} to {date_end}')

In [ ]:
full = ((6510, 6624), 'Full')
Ha = ((6548, 6578), 'Hα')
HeI = ((6648, 6708), 'He I')

# shift = 9.69 #69 Ori
# shift = 5.6 #Bet CMi
# shift = 5 #Pi Aqr
shift = 5.75

plot_bess_spectra(
    bess_files_sorted=bess_files_sorted,
    target_name=target_name,
    offset=0.15,
    wavelength_range=full,
    y_shift = shift,
    save_fig = True
)


plot_bess_spectra(
    bess_files_sorted=bess_files_sorted,
    target_name=target_name,
    offset=0.15,
    wavelength_range=Ha,
    y_shift = shift,
    save_fig = True
)


# plot_bess_spectra(
#     bess_files_sorted=bess_files_sorted,
#     target_name=target_name,
#     offset=0.15,
#     wavelength_range=HeI,
#     y_shift = shift,
#     save_fig = True
# )


In [ ]:
# # plot_range = ('2019-01-01', '2019-03-01')
# plot_range = None

plot_bess_dynamic_spectrum(
    bess_files_sorted=bess_files_sorted,
    target_name=target_name,
    wavelength_range=full,
    cmap='turbo',
    save_fig = True
)

plot_bess_dynamic_spectrum(
    bess_files_sorted=bess_files_sorted,
    target_name=target_name,
    wavelength_range=Ha,
    cmap='turbo',
    save_fig = True
)

# plot_bess_dynamic_spectrum(
#     bess_files_sorted=bess_files_sorted,
#     target_name=target_name,
#     wavelength_range=HeI,
#     cmap='turbo',
#     save_fig = True
# )

In [ ]:
import plotly.graph_objects as go
import numpy as np

def plot_stacked_spectra_plotly(all_spectra, y_shift=1.0, title="Stacked Spectra (Interactive)"):
    """
    Interactive stacked spectral viewer using Plotly.

    Parameters
    ----------
    all_spectra : list of dict
        Each element must contain:
            'file' : filename
            'wave' : wavelength array
            'flux' : flux array
            'date' : numeric date (MJD or JD)
            'date_str' : readable string
    y_shift : float
        Vertical offset between spectra.
    """

    fig = go.Figure()

    # Normalize date spacing
    dates = np.array([s["date"] for s in all_spectra])
    min_date = dates.min()
    date_offsets = (dates - min_date) / (dates.max() - min_date + 1e-6)

    for i, spec in enumerate(all_spectra):
        w = spec["wave"]
        f = spec["flux"]

        # Vertical offset
        offset = date_offsets[i] * y_shift

        fig.add_trace(go.Scatter(
            x=w,
            y=f + offset,
            mode='lines',
            name=spec["date_str"],
            hovertemplate=(
                "λ = %{x}<br>"
                "Flux = %{y}<br>"
                f"Date = {spec['date_str']}<extra></extra>"
            )
        ))

    fig.update_layout(
        title=title,
        xaxis_title="Wavelength (Å)",
        yaxis_title="Flux + Offset",
        hovermode="x unified",
        template="plotly_white",
        showlegend=True,
        height=800
    )

    fig.show()

In [ ]:
from scipy.signal import medfilt
from scipy.optimize import curve_fit

def continuum_normalize(wavelengths, spectrum, order=3, filt=51):
    # Smooth to suppress lines
    smoothed = medfilt(spectrum, filt)
    # Fit polynomial to smoothed continuum
    p = np.polyfit(wavelengths, smoothed, order)
    continuum = np.polyval(p, wavelengths)
    return spectrum / continuum

def spec_grid(target, radius, bess_files_sorted, spec_feature=6562.8):
    """
    Extract spectral segments around Hα from 1-D spectra.

    Args:
        target: placeholder, not used here
        radius: number of pixels on either side of target wavelength
        bess_files_sorted: list of FITS file paths
        spec_feature: target rest wavelength

    Returns:
        result: concatenated np.ndarray of segments
    """
    result = np.array([])

    for fname in bess_files_sorted:
        wavelengths, spectrum, header = BeSSSpectra(fname)
        if wavelengths is None:
            continue

        # Find closest pixel to Hα
        norm_spectrum  = continuum_normalize(wavelengths, spectrum)
        index = np.argmin(np.abs(wavelengths - spec_feature))
        if index - radius < 0 or index + radius > len(norm_spectrum):
            continue

        segment = norm_spectrum[index - radius:index + radius]
        result = np.append(result, segment)

    return result

# Alternative Plotting

In [ ]:
def bin_times_and_spectra(dates, flux_arrays, bin_days):
    """
    Bin spectra into fixed time bins.

    Parameters
    ----------
    dates : list of datetime objects
    flux_arrays : list of 1D numpy arrays (already interpolated to common grid)
    bin_days : int
        Bin width in days.

    Returns
    -------
    binned_dates : list of datetime (bin centers)
    binned_fluxes : list of 1D arrays (averaged per bin)
        Missing-data bins are filled with NaNs (blank rows).
    """
    if len(dates) == 0:
        return [], []

    # Sort by date
    sorted_idx = np.argsort(dates)
    dates = np.array([dates[i] for i in sorted_idx])
    flux_arrays = [flux_arrays[i] for i in sorted_idx]

    # Build bins
    start_date = dates[0]
    end_date = dates[-1]

    delta = timedelta(days=bin_days)
    bins = []
    t = start_date
    while t <= end_date + delta:
        bins.append(t)
        t += delta

    binned_dates = []
    binned_fluxes = []

    for i in range(len(bins) - 1):
        t0 = bins[i]
        t1 = bins[i + 1]
        mid = t0 + (t1 - t0) / 2

        # indices of spectra in this bin
        idx = np.where((dates >= t0) & (dates < t1))[0]

        if len(idx) == 0:
            # blank bin → all NaNs
            blank = np.full_like(flux_arrays[0], np.nan)
            binned_dates.append(mid)
            binned_fluxes.append(blank)
        else:
            # average available spectra
            stacked = np.vstack([flux_arrays[k] for k in idx])
            avg = np.nanmean(stacked, axis=0)
            binned_dates.append(mid)
            binned_fluxes.append(avg)

    return binned_dates, binned_fluxes

def plot_bess_spectra_binned(
    bess_files_sorted,
    target_name,
    offset=0.2,
    wavelength_range=(6510, 6624),
    y_shift=2,
    save_fig=True,
    bin_days=7
):
    """
    Stacked BeSS spectra with temporal binning.

    Each bin of bin_days produces one averaged spectrum.
    Missing bins produce blank rows.
    """

    all_spectra = []
    all_dates = []

    # --- Collect spectra ---
    for fname in bess_files_sorted:
        try:
            wavelengths, spectrum, header = BeSSSpectra(fname)
            date_obs = header.get('DATE-OBS', None)

            if not date_obs:
                continue

            try:
                date_obj = datetime.strptime(date_obs[:10], "%Y-%m-%d")
            except ValueError:
                continue

            spectrum = spectrum / np.nanmax(spectrum)

            all_spectra.append((date_obj, wavelengths, spectrum))

        except Exception as e:
            print(f"⚠️ Skipping {fname}: {e}")

    if len(all_spectra) == 0:
        print("❌ No spectra found.")
        return

    # --- Build common wavelength grid ---
    wmin, wmax = wavelength_range[0]
    common_wave = np.linspace(wmin, wmax, 1000)

    dates = []
    flux_arrays = []

    for date_obj, wave, flux in all_spectra:
        mask = (wave >= wmin) & (wave <= wmax)
        if np.sum(mask) < 5:
            continue

        interp_flux = interp1d(
            wave[mask], flux[mask], kind='linear',
            bounds_error=False, fill_value=np.nan
        )(common_wave)

        dates.append(date_obj)
        flux_arrays.append(interp_flux)

    # --- Bin the spectra ---
    binned_dates, binned_fluxes = bin_times_and_spectra(dates, flux_arrays, bin_days)

    # --- Plot ---
    plt.figure(figsize=(10, 0.5 * len(binned_dates) + 5))

    for j, flux in enumerate(binned_fluxes):
        plt.plot(common_wave, flux + (j - y_shift) * offset, color='black', lw=0.8)

    plt.xlim(*wavelength_range[0])
    plt.xlabel("Wavelength (Å)")
    plt.ylabel("Binned Date + Normalized Flux Offset")
    plt.title(f"Stacked BeSS Spectra (binned by {bin_days} days)")

    y_ticks = np.arange(len(binned_dates)) * offset
    y_labels = [d.strftime("%Y-%m-%d") for d in binned_dates]
    plt.yticks(y_ticks, y_labels)

    plt.tight_layout()

    if save_fig:
        out_folder = os.path.expanduser(
            f'~/gdrive/Shared drives/MACRO-Be/resources/BeSS Variability Plots/{target_name}/'
        )
        os.makedirs(out_folder, exist_ok=True)
        out_path = out_folder + f'{target_name}_{wavelength_range[1]}_binned_{bin_days}d.png'
        plt.savefig(out_path, dpi=300)
        print(f"Saved: {out_path}")

    plt.show()

def plot_bess_dynamic_spectrum_binned(
    bess_files_sorted,
    target_name,
    wavelength_range=(6510, 6624),
    date_range=None,
    cmap='turbo',
    save_fig=True,
    bin_days=7
    ):
        """
        Dynamic spectrum with fixed temporal binning.
        """

        wmin, wmax = wavelength_range[0]
        common_wave = np.linspace(wmin, wmax, 1000)

        all_dates = []
        all_fluxes = []

        # Optional date filtering
        if date_range is not None:
            start_date = datetime.strptime(date_range[0], "%Y-%m-%d")
            end_date = datetime.strptime(date_range[1], "%Y-%m-%d")

        # --- Load spectra ---
        for fname in bess_files_sorted:
            try:
                wave, flux, header = BeSSSpectra(fname)
                date_obs = header.get('DATE-OBS')
                if not date_obs:
                    continue

                date_obj = datetime.strptime(date_obs[:10], "%Y-%m-%d")

                if date_range and (date_obj < start_date or date_obj > end_date):
                    continue

                mask = (wave >= wmin) & (wave <= wmax)
                if np.sum(mask) < 5:
                    continue

                flux = flux / np.nanmax(flux)
                interp_flux = interp1d(
                    wave[mask], flux[mask], kind='linear',
                    bounds_error=False, fill_value=np.nan
                )(common_wave)

                all_dates.append(date_obj)
                all_fluxes.append(interp_flux)

            except Exception as e:
                print(f"⚠️ Skipping {fname}: {e}")

        if len(all_fluxes) == 0:
            print("❌ No spectra found in range.")
            return

        # --- Bin data ---
        binned_dates, binned_fluxes = bin_times_and_spectra(all_dates, all_fluxes, bin_days)

        flux_matrix = np.vstack(binned_fluxes)

        y_positions = np.arange(len(binned_dates))

        # --- Plot ---
        plt.figure(figsize=(10, 5 + 0.25 * len(binned_dates)))

        plt.imshow(
            flux_matrix,
            aspect='auto',
            cmap=cmap,
            extent=[common_wave[0], common_wave[-1], y_positions[0], y_positions[-1]],
            origin='lower'
        )

        plt.yticks(y_positions, [d.strftime("%Y-%m-%d") for d in binned_dates])
        plt.xlabel("Wavelength (Å)")
        plt.ylabel(f"Time (binned every {bin_days} days)")
        plt.title(f"Dynamic Spectrum for {target_name}")

        plt.colorbar(label="Normalized Flux")
        plt.tight_layout()

        if save_fig:
            out_folder = os.path.expanduser(
                f'~/gdrive/Shared drives/MACRO-Be/resources/BeSS Variability Plots/{target_name}/'
            )
            os.makedirs(out_folder, exist_ok=True)
            out_path = out_folder + f'{target_name}_{wavelength_range[1]}_dynamic_binned_{bin_days}d.png'
            plt.savefig(out_path, dpi=300)
            print(f"Saved: {out_path}")

        plt.show()

In [ ]:
# # plot_range = ('2019-01-01', '2019-03-01')
# plot_range = None

full = ((6510, 6624), 'Full')
Ha = ((6548, 6578), 'Hα')
HeI = ((6648, 6708), 'He I')

plot_bess_dynamic_spectrum_binned(
    bess_files_sorted=bess_files_sorted,
    target_name=target_name,
    wavelength_range=full,
    cmap='turbo',
    bin_days=7,
    save_fig = True
)

plot_bess_dynamic_spectrum_binned(
    bess_files_sorted=bess_files_sorted,
    target_name=target_name,
    wavelength_range=Ha,
    cmap='turbo',
    bin_days=7,
    save_fig = True
)

# plot_bess_dynamic_spectrum_binned(
#     bess_files_sorted=bess_files_sorted,
#     target_name=target_name,
#     wavelength_range=HeI,
#     cmap='turbo',
#     save_fig = True
# )